In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
from pathlib import Path
import json
import pandas as pd

project_root = Path.cwd()
climate_path = project_root / "dataset" / "climate" / "average_all.psv"
charging_path = project_root / "dataset" / "charging" / "caltech_sessions.json"

climate_df = pd.read_csv(climate_path, sep="|")
with charging_path.open("r", encoding="utf-8") as file_handle:
    charging_data = json.load(file_handle)

climate_df.head()

,Year,Month,Day,Hour,temperature_mean,dew_point_temperature_mean,station_level_pressure_mean,wind_speed_mean,precipitation_mean,relative_humidity_mean,visibility_mean,snow_depth_mean,temperature_std,dew_point_temperature_std,station_level_pressure_std,wind_speed_std,precipitation_std,relative_humidity_std,visibility_std,snow_depth_std
0,2018,1,1,0,16.733333,8.500000,1001.25,1.733333,0.0,60.333333,10.728667,NaN,1.553491,3.148015,14.071425,1.501111,0.0,17.502381,4.916558,NaN
1,2018,1,1,1,14.833333,8.700000,1002.10,1.366667,0.0,68.000000,10.728667,NaN,0.862168,3.251154,14.283557,1.305118,0.0,16.093477,4.916558,NaN
2,2018,1,1,2,13.900000,11.266667,1002.80,1.400000,0.0,84.333333,10.192333,NaN,0.556776,0.378594,14.283557,1.212436,0.0,4.932883,5.651619,NaN
3,2018,1,1,3,13.000000,10.671429,1009.10,0.428571,0.0,86.285714,7.931429,NaN,0.223607,1.307305,8.833742,0.731925,0.0,6.969321,5.640841,NaN
4,2018,1,1,4,12.200000,10.850000,1003.10,0.000000,0.0,91.500000,5.230000,NaN,0.000000,0.353553,14.283557,0.000000,0.0,2.121320,1.706956,NaN


In [5]:
import pandas as pd

items_df = pd.DataFrame(charging_data.get("_items", [])).head(3)
items_df["connection_dt"] = pd.to_datetime(
    items_df["connectionTime"], format="%a, %d %b %Y %H:%M:%S GMT", utc=True
 )
items_df["Year"] = items_df["connection_dt"].dt.year
items_df["Month"] = items_df["connection_dt"].dt.month
items_df["Day"] = items_df["connection_dt"].dt.day
items_df["Hour"] = items_df["connection_dt"].dt.hour

climate_keyed = climate_df.copy()
for col in ["Year", "Month", "Day", "Hour"]:
    climate_keyed[col] = pd.to_numeric(climate_keyed[col], errors="coerce").astype("Int64")

matched = items_df.merge(
    climate_keyed, on=["Year", "Month", "Day", "Hour"], how="left"
 )

matched[["sessionID", "connectionTime", "temperature_mean", "wind_speed_mean"]]

,sessionID,connectionTime,temperature_mean,wind_speed_mean
0,2_39_78_362_2018-04-25 11:08:04.400812,"Wed, 25 Apr 2018 11:08:04 GMT",12.7,0.750000
1,2_39_95_27_2018-04-25 13:45:09.617470,"Wed, 25 Apr 2018 13:45:10 GMT",12.6,2.066667
2,2_39_79_380_2018-04-25 13:45:49.962001,"Wed, 25 Apr 2018 13:45:50 GMT",12.6,2.066667
